# Manual ARIMA Model

## Purpose
Fits a manually specified **ARIMA(1,1,1)** model using walk-forward validation as an initial proof-of-concept. Seasonal differencing (lag-12) is applied manually before each fit, and the forecast is inverted back to the original scale.

This notebook precedes the automated grid search and serves to validate the walk-forward pipeline before scaling it across a large parameter space.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
-The walk-forward predictions and RMSE.

In [1]:
import pandas as pd
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from math import sqrt

## Differencing and Inverse-Difference Helpers

`difference()` computes a lag-`interval` difference to remove seasonal structure before passing the series to ARIMA.

`inverse_difference()` inverts the forecast back to the original scale by adding the differenced prediction to the observation that was `interval` steps back in history. This must be applied after every ARIMA forecast.

In [2]:
# create a differenced series
def difference(dataset, interval=1):
    diff = list()
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return diff

In [3]:
# invert differenced value
def inverse_difference(history, yhat, interval=1):
    return yhat + history[-interval]

## Load Training Data

In [4]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Name: Sales, dtype: int64

## Train / Test Split

50/50 split, consistent with the persistence baseline notebook, enabling a direct RMSE comparison.

In [5]:
# prepare data
X = series.values
X = X.astype('float64')
train_size = int(len(X) * 0.50)
train, test = X[0:train_size], X[train_size:]

## Walk-Forward Validation with ARIMA(1,1,1)

At each step:
1. Apply a 12-month seasonal difference to the current history window.
2. Fit an **ARIMA(1,1,1)** on the differenced series.
3. Forecast one step ahead and invert the seasonal difference to recover the original scale.
4. Append the true observation to history before moving to the next step.

The model is re-fit from scratch at every step — this is computationally expensive but ensures no look-ahead bias.

In [6]:
# walk-forward validation
history = [x for x in train]
predictions = list()
for i in range(len(test)):
    # difference data
    months_in_year = 12
    diff = difference(history, months_in_year)
    # predict
    model = ARIMA(diff, order=(1,1,1))
    model_fit = model.fit()
    yhat = model_fit.forecast()[0]
    yhat = inverse_difference(history, yhat, months_in_year)
    predictions.append(yhat)
    # observation
    obs = test[i]
    history.append(obs)
    print('>Predicted=%.3f, Expected=%.3f' % (yhat, obs))

>Predicted=8076.988, Expected=8314.000
>Predicted=9747.154, Expected=10651.000
>Predicted=5994.371, Expected=3633.000
>Predicted=3820.284, Expected=4292.000
>Predicted=4041.968, Expected=4154.000
>Predicted=4990.405, Expected=4121.000
>Predicted=5129.641, Expected=4647.000
>Predicted=5031.196, Expected=4753.000
>Predicted=4133.285, Expected=3965.000
>Predicted=2095.321, Expected=1723.000
>Predicted=5216.271, Expected=5048.000
>Predicted=5866.317, Expected=6922.000
>Predicted=8591.067, Expected=9858.000
>Predicted=11028.649, Expected=11331.000
>Predicted=4090.352, Expected=4016.000
>Predicted=4767.109, Expected=3957.000
>Predicted=4656.326, Expected=4510.000
>Predicted=4577.708, Expected=4276.000
>Predicted=5108.656, Expected=4968.000
>Predicted=5202.831, Expected=4677.000
>Predicted=4423.982, Expected=3523.000
>Predicted=2162.388, Expected=1821.000
>Predicted=5463.233, Expected=5222.000
>Predicted=7331.345, Expected=6872.000
>Predicted=10258.650, Expected=10803.000
>Predicted=11732.476

## Report RMSE

Compare this RMSE against the persistence baseline. An improvement confirms that ARIMA structure adds predictive value over the naïve model.

In [7]:
# report performance
rmse = sqrt(mean_squared_error(test, predictions))
print('RMSE: %.3f' % rmse)

RMSE: 961.619
